# Allen Cell Types — Data Loading & Initial Exploration

**Dataset:** Allen Institute Cell Types Database (CRCNS)  
**Reference:** https://alleninstitute.github.io/AllenSDK/cell_types.html

**Purpose:** Ground-truth cell type reference for validating spikeparam waveform features.
~1,920 mouse cells with known dendrite type (spiny = PC, aspiny = IN), transgenic line,
cortical layer, and brain area. All recordings are whole-cell patch clamp with injected
current (Long Square, Ramp, Noise) — analogous to pvc-6, NOT to spe-1 (which is spontaneous).

**Pipeline:**
1. Load cell metadata
2. Spot-check a single cell: traces, spike detection, waveform gallery
3. Run batch feature extraction → `scripts/run_allen_ct_batch.py`
4. Load and inspect the resulting population features pickle
5. Save metadata pickle for downstream analyses


In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

sys.path.insert(0, os.path.abspath('allen_ct_helper_modules'))

from allen_ct_helper_modules.data_loader import (
    load_cell_metadata,
    load_precomputed_features,
    load_morphology_features,
    load_merged_dataset,
    get_long_square_sweeps,
    load_voltage_trace,
    save_to_pickle,
    load_from_pickle,
)
from allen_ct_helper_modules.config import (
    ALLEN_CT_PICKLE_ROOT,
    PRIMARY_SPIKE_FEATURES,
)

## 1. Load Cell Metadata

Download metadata for all mouse cells from the Allen Cell Types Database.
Key columns: `id`, `dendrite_type` (spiny/aspiny), `structure_area_abbrev`, 
`structure_layer_name`, `transgenic_line`, `normalized_depth`.

In [ ]:
cells_df = load_cell_metadata(species='Mus musculus')
print(f'{len(cells_df)} cells loaded')
cells_df.head()

In [ ]:
# Quick metadata overview
print('=== Dendrite type ===')
print(cells_df['dendrite_type'].value_counts())
print()
print('=== Brain area (top 10) ===')
print(cells_df['structure_area_abbrev'].value_counts().head(10))
print()
print('=== Transgenic lines (top 10) ===')
print(cells_df['transgenic_line'].value_counts().head(10))

## 2. Spot-Check: Single Cell

Before running the full batch, verify the loading pipeline on one cell:
list available sweeps, plot the voltage trace, confirm spike detection works.

In [ ]:
# Pick first cell — change index to inspect others
specimen_id = int(cells_df.iloc[0]['id'])
print(f'Specimen ID: {specimen_id}')
print(cells_df.iloc[0][['structure_area_abbrev', 'dendrite_type', 'transgenic_line']])

In [ ]:
# List available Long Square sweeps
sweeps_meta = get_long_square_sweeps(specimen_id)
print(f'{len(sweeps_meta)} Long Square sweeps found')
pd.DataFrame(sweeps_meta)[['sweep_number', 'stimulus_name', 'stimulus_absolute_amplitude', 'num_spikes']].head(10)

### Load and plot one sweep

Verify signal quality and stimulus delivery before committing to the full batch download.

In [ ]:
# Load a suprathreshold sweep (pick one with spikes from the table above)
sweep_idx = 5
voltage, stimulus, times, fs, idx_range = load_voltage_trace(specimen_id, sweeps_meta[sweep_idx])

print(f'Sampling rate: {fs} Hz')
print(f'Trace length: {len(voltage)/fs:.2f} s ({len(voltage)} samples)')
print(f'Voltage range: {voltage.min():.1f} to {voltage.max():.1f} mV')
print(f'Stimulus amplitude: {sweeps_meta[sweep_idx].get("stimulus_absolute_amplitude", "N/A")} pA')

start, stop = idx_range
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
axes[0].plot(times[start:stop], voltage[start:stop], 'k', linewidth=0.8)
axes[0].set_ylabel('Voltage (mV)')
axes[0].set_title(f'Specimen {specimen_id} — Sweep {sweeps_meta[sweep_idx]["sweep_number"]}')
axes[1].plot(times[start:stop], stimulus[start:stop], 'steelblue', linewidth=0.8)
axes[1].set_ylabel('Stimulus (pA)')
axes[1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

### Verify spike detection

Same threshold-crossing approach used in spe-1 and pvc-6.

In [ ]:
v_trim = voltage[start:stop]
t_trim = times[start:stop]
one_ms = int(fs / 1000)

thresh_mv = -10  # mV — same as spe-1 / pvc-6
idx_spikes, props = find_peaks(v_trim, height=thresh_mv, distance=one_ms)
print(f'Detected {len(idx_spikes)} spikes  |  amplitudes: {props["peak_heights"].min():.1f}–{props["peak_heights"].max():.1f} mV')

plt.figure(figsize=(14, 4))
plt.plot(t_trim, v_trim, 'k', linewidth=0.8)
plt.plot(t_trim[idx_spikes], v_trim[idx_spikes], 'r.', markersize=8, label='spikes')
plt.axhline(thresh_mv, color='gray', linestyle='--', linewidth=0.8, label=f'threshold ({thresh_mv} mV)')
plt.xlabel('Time (ms)')
plt.ylabel('Voltage (mV)')
plt.legend()
plt.tight_layout()
plt.show()

### Spike waveform gallery

Extract ±10 ms windows around detected spikes to visually confirm waveform quality
before committing to batch spikeparam feature extraction.

In [ ]:
window_pre  = int(one_ms * 10)
window_post = int(one_ms * 10)

waveforms = []
for idx in idx_spikes:
    if idx - window_pre >= 0 and idx + window_post < len(v_trim):
        waveforms.append(v_trim[idx - window_pre : idx + window_post])
waveforms = np.array(waveforms)
wt = np.linspace(-10, 10, waveforms.shape[1])

print(f'Extracted {len(waveforms)} waveforms')
plt.figure(figsize=(8, 5))
for w in waveforms:
    plt.plot(wt, w, 'k', alpha=0.3, linewidth=0.5)
plt.plot(wt, waveforms.mean(axis=0), 'r', linewidth=2, label='mean')
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Time from peak (ms)')
plt.ylabel('Voltage (mV)')
plt.title(f'Specimen {specimen_id} — {len(waveforms)} spike waveforms')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Batch Feature Extraction

Run the batch script to extract spikeparam features for all cells.
This downloads NWB files (~100–300 GB total) and runs the same spikeparam
pipeline used for spe-1 and pvc-6.

**Run from the repo root:**
```bash
# Test first (10 cells):
python scripts/run_allen_ct_batch.py --n-cells 10

# All cells (overnight job):
python scripts/run_allen_ct_batch.py --workers 4
```

Output:
- Per-cell: `allen_ct_pickles/features/{specimen_id}_features.pkl`
- Population: `allen_ct_pickles/allen_ct_population_features.pkl`

## 4. Load Population Features (post-batch)

Once the batch has completed, load the merged population DataFrame.
This is the primary input to all downstream population analyses.

In [ ]:
pop_path = os.path.join(ALLEN_CT_PICKLE_ROOT, 'allen_ct_population_features.pkl')

if os.path.exists(pop_path):
    pop_df = pd.read_pickle(pop_path)
    print(f'Population DataFrame: {pop_df.shape}')
    print(f'Cells: {pop_df["specimen_id"].nunique()}')
    print(f'Spikes: {len(pop_df)}')
    print()
    print('Columns:', pop_df.columns.tolist())
    pop_df.head()
else:
    print(f'Population pickle not found at {pop_path}')
    print('Run the batch script first:  python scripts/run_allen_ct_batch.py')

## 5. Save Metadata & Merged Dataset Pickles

Save the cell metadata and full merged dataset (metadata + precomputed ephys + morphology)
for use in downstream analyses and the dataset summary notebook.

In [ ]:
os.makedirs(ALLEN_CT_PICKLE_ROOT, exist_ok=True)

# Cell metadata
meta_path = os.path.join(ALLEN_CT_PICKLE_ROOT, 'allen_ct_cells_df.pkl')
cells_df.to_pickle(meta_path)
print(f'Saved metadata ({len(cells_df)} cells) → {meta_path}')

# Full merged dataset: metadata + precomputed ephys + morphology
merged_path = os.path.join(ALLEN_CT_PICKLE_ROOT, 'allen_ct_merged_dataset.pkl')
merged_df = load_merged_dataset(species='Mus musculus', include_morphology=True)
merged_df.to_pickle(merged_path)
print(f'Saved merged dataset {merged_df.shape} → {merged_path}')